# 005 — Vectores, matrices y geometría para IA

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

Un vector x ∈ ℝⁿ es a la vez lista de números, punto/flecha y **representación de un
objeto** (fila de datos, imagen, embedding). La apuesta del deep learning: la semántica se
codifica como geometría — cosas parecidas, vectores cercanos.

```text
producto punto:  x·y = Σ xᵢyᵢ          (mide alineación; la neurona computa w·x + b)
norma:           ‖x‖ = √(x·x)
coseno:          cos θ = x·y / (‖x‖‖y‖)  (similitud que ignora magnitud)
distancia:       d(x,y) = ‖x−y‖          (sensible a magnitud)
```

Una matriz A ∈ ℝᵐˣⁿ es una **transformación lineal**: sus columnas dicen a dónde van los
ejes; el producto de matrices es composición y **no conmuta**. Una capa densa es
`g(Wx + b)`; sin la no linealidad g, apilar capas colapsa a una sola matriz.

Ejemplo trabajado de la teoría: con d₁=[2,3,0], d₂=[1,2,0], d₃=[0,1,4],
cos(d₁,d₂) ≈ 0.992 (mismo tema) y cos(d₁,d₃) ≈ 0.202 — pero ‖d₁−d₂‖ = √2 > 0: el coseno
corrige la diferencia de longitud de los documentos, la euclídea no.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** d₂ = 2·d₁, por lo que cos(d₁,d₂) = **1.0** exacto (misma dirección) aunque
‖d₁−d₂‖ = ‖d₁‖ = √18 ≈ 4.24 sea grande. cos(d₁,d₃) = (4+4+0)/(√18·√17) ≈ 0.457;
‖d₁−d₃‖ = √(9+9+1) = √19 ≈ 4.36. El coseno declara a (d₁,d₂) idénticos en tema; la
euclídea los declara tan lejanos como (d₁,d₃): la medida correcta depende de si la
magnitud (longitud del documento) es señal o ruido.

**Ejercicio 2.** w·x+b = 2−2+1−1 = 0 → no se activa (no supera 0). La entrada unitaria que
maximiza w·x es w/‖w‖ = [2,−1,0.5]/√5.25 ≈ [0.873,−0.436,0.218]: la neurona es un detector
de su propio patrón de pesos.

**Ejercicio 3.** AB·[1,0]: primero B estira → [2,0], luego A rota → [0,2].
BA·[1,0]: primero A rota → [0,1], luego B estira x → [0,1]. Resultados distintos ([0,2] vs
[0,1]): estirar un eje y luego rotarlo no es lo mismo que rotar primero (el estiramiento
actúa sobre ejes distintos).

**Ejercicio 4.** Con distinta semilla cambia la inicialización (dónde empieza el punto) y
por tanto la trayectoria; la superficie que se optimiza es la misma. Si el problema es
convexo, el mínimo alcanzado coincide; con superficies no convexas, distintas semillas
pueden terminar en distintos mínimos locales — por eso se reporta sobre varias semillas.

In [ ]:
result = run_lab("optimization", seed=5)
assert result["kind"] == "optimization"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1-3 — verificación numérica
import math

def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

def norm(a):
    return math.sqrt(dot(a, a))

def cos(a, b):
    return dot(a, b) / (norm(a) * norm(b))

d1, d2, d3 = [1, 4, 1], [2, 8, 2], [4, 1, 0]
print(f"cos(d1,d2)={cos(d1,d2):.3f}  cos(d1,d3)={cos(d1,d3):.3f}")
print(f"dist(d1,d2)={norm([a-b for a,b in zip(d1,d2)]):.3f}  dist(d1,d3)={norm([a-b for a,b in zip(d1,d3)]):.3f}")

# Ejercicio 2
w, x, b = [2, -1, 0.5], [1, 2, 2], -1
print("activación:", dot(w, x) + b)  # 0.0 → no supera el umbral

# Ejercicio 3 — composición no conmutativa
def matvec(M, v):
    return [M[0][0]*v[0] + M[0][1]*v[1], M[1][0]*v[0] + M[1][1]*v[1]]

A = [[0, -1], [1, 0]]   # rotación 90°
B = [[2, 0], [0, 1]]    # estirar eje x
v = [1, 0]
print("A(B v) =", matvec(A, matvec(B, v)))  # [0, 2]
print("B(A v) =", matvec(B, matvec(A, v)))  # [0, 1]

## Reflexión

1. El laboratorio `optimization` mueve un punto en un espacio vectorial. ¿Qué papel juegan
   la norma y la dirección del paso en lo que observaste en el JSON?
2. ¿En qué caso concreto de un sistema de búsqueda semántica preferirías distancia euclídea
   sobre coseno, y qué tendría que ser cierto sobre las normas de los embeddings?
3. Explica sin fórmulas, a un colega no técnico, por qué apilar tres capas lineales sin
   activación no da un modelo más expresivo que una sola.